# Feature Engineering — Shared Source of Truth

This notebook is the **single source of truth** for the 85-feature engineering
pipeline used by:

* [predict_betting.ipynb](predict_betting.ipynb) — production inference (one upcoming week)
* [model_comparison.ipynb](model_comparison.ipynb) — training / backtesting (all historical games)

Both consumer notebooks load this one via `json.load + exec` over every code cell, so the
definitions live here exactly once. The two computations are equivalent at the
target row: training uses `shift(1).rolling(5)` per team-game, inference uses
`rolling(5)` then `nth(-1)` — both yield the same value for the row going into
the next game.

**Convention** (matches the rest of the repo): each section is **markdown →
code → inline test**. The markdown states purpose / inputs / outputs / what the
test asserts. Tests are guarded by `RUN_TESTS` so they run when this notebook
is executed standalone (development) but can be skipped by consumers that set
`RUN_TESTS = False` before loading.

**Public surface** (names available after the loader runs):

| Name | Type |
|------|------|
| `TEAM_MAP` | dict — historical → modern team-abbrev map |
| `FEATURE_COLS_85` | list — canonical 85 engineered feature names |
| `PROD_FEATURES_35` | list — top-35 production subset (feature ablation 2026-05-20) |
| `norm_name(s)` | fn — name normaliser for AllPro / injury joins |
| `canonicalize_ngs_team(t, s)` | fn — NGS team-abbrev → schedule per season |
| `build_features(...)` | fn — main inference entrypoint |
| `build_numeric_features(...)` | fn — categorical encode + numeric matrix |
| `_build_*` | per-group helpers, prefixed underscore (debugging only) |


## Parameters

`RUN_TESTS` controls whether the inline test cells execute. Default `True` for
standalone development. Consumer notebooks set this to `False` in their own
namespace before loading this notebook to skip the synthetic-data tests
(which take ~1 second and re-cover what production data validates anyway).


In [ ]:
# Default ON for standalone runs. Consumer notebooks set this False before loading.
RUN_TESTS = globals().get("RUN_TESTS", True)


## Imports

Standard scientific Python plus `nflreadpy` for the live API calls in
`_build_qb_switch`, `_build_passer_rating`, `_build_injuries`, and
`_build_coach_win_pct` (each falls back to the caller-supplied DataFrame on
failure).


In [ ]:
import re
import unicodedata
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import nflreadpy as nfl


In [ ]:
# Smoke-test: verify the imports resolve. Asserts always run (cheap, catches
# missing dependencies); the success print is gated so production loaders stay quiet.
assert callable(getattr(nfl, "load_nextgen_stats", None)), "nflreadpy.load_nextgen_stats missing"
assert callable(getattr(nfl, "load_injuries", None)),       "nflreadpy.load_injuries missing"
assert callable(getattr(nfl, "load_schedules", None)),      "nflreadpy.load_schedules missing"
if RUN_TESTS:
    print("✓ Imports loaded.")


## Constants

* **`TEAM_MAP`** — folds historical / alternate NFL team abbreviations into the
  modern set so AllPro CSV and PBP tables join cleanly with the schedule.
* **`FEATURE_COLS_85`** — the canonical ordered list of all 85 engineered
  features, exactly matching the column names the production pkls were trained on.
  The trailing space in `"allpro_diff_home_def_away_off_3_years "` is **intentional**
  and must not be stripped without retraining.
* **`PROD_FEATURES_35`** — top-35 subset by combined XGB gain + Ridge |coef| +
  LGB gain, picked by the 2026-05-20 ablation study. Engineering still produces
  all 85; only these 35 are passed to model training.


In [ ]:
TEAM_MAP = {
    # Modern moves
    "STL": "LA",  "LAR": "LA",   "OAK": "LV",  "LVR": "LV",
    "SD":  "LAC", "SDG": "LAC",
    # Pro-Football-Reference 3-letter codes → schedule 2-letter codes
    "NWE": "NE",  "KAN": "KC",   "GNB": "GB",  "NOR": "NO",
    "TAM": "TB",  "SFO": "SF",
    # Pre-2002 / alternate abbreviations in historical AllPro CSV
    "ARZ": "ARI", "BLT": "BAL",  "CLV": "CLE", "HST": "HOU", "JAC": "JAX",
}

FEATURE_COLS_85 = [
    "roof",
    "surface",
    "spread_line",
    "away_rest",
    "home_rest",
    "total_line",
    "div_game",
    "home_rolling_avg_epa",
    "home_rolling_avg_yards",
    "home_rolling_play_count",
    "away_rolling_avg_epa",
    "away_rolling_avg_yards",
    "away_rolling_play_count",
    "home_rolling_allowed_avg_epa",
    "home_rolling_allowed_avg_yards",
    "home_rolling_allowed_play_count",
    "away_rolling_allowed_avg_epa",
    "away_rolling_allowed_avg_yards",
    "away_rolling_allowed_play_count",
    "epa_home_off_away_def_rolling_diff",
    "epa_home_def_away_off_rolling_diff",
    "avg_yards_home_off_away_def_rolling_diff",
    "avg_yards_home_def_away_off_rolling_diff",
    "play_count_home_off_away_def_rolling_diff",
    "play_count_home_def_away_off_rolling_diff",
    "home_recent_sos_opponent_avg",
    "home_season_sos_opponent_avg",
    "away_recent_sos_opponent_avg",
    "away_season_sos_opponent_avg",
    "sos_diff",
    "season_sos_diff",
    "home_allpro_last_3_years_weighted",
    "away_allpro_last_3_years_weighted",
    "diff_allpro_last_3_years_weighted",
    "home_allpro_prev_year",
    "away_allpro_prev_year",
    "diff_allpro_prev_year",
    "home_offense_allpro_3_years",
    "away_offense_allpro_3_years",
    "home_defense_allpro_3_years",
    "away_defense_allpro_3_years",
    "allpro_diff_home_off_away_def_3_years",
    "allpro_diff_home_def_away_off_3_years ",  # trailing space matches model
    "home_offense_allpro_prev_year",
    "away_offense_allpro_prev_year",
    "home_defense_allpro_prev_year",
    "away_defense_allpro_prev_year",
    "allpro_diff_home_off_away_def_prev_year",
    "allpro_diff_home_def_away_off_prev_year",
    "league_rolling_avg_abs_margin_by_week",
    "home_pr_prev_year",
    "away_pr_prev_year",
    "diff_pr_prev_year",
    "home_cpae_prev_year",
    "away_cpae_prev_year",
    "diff_cpae_prev_year",
    "home_time_to_throw_prev_year",
    "away_time_to_throw_prev_year",
    "diff_time_to_throw_prev_year",
    "home_injured_count",
    "away_injured_count",
    "diff_injured_count",
    "diff_active_allpro_weighted",
    "diff_active_allpro_prev_year",
    "home_rolling_win_pct",
    "away_rolling_win_pct",
    "sack_diff",
    "sack_diff_reverse",
    "turnover_diff",
    "turnover_diff_reverse",
    "third_down_diff",
    "third_down_diff_reverse",
    "cover_rate_diff",
    "scoring_diff",
    "scoring_diff_reverse",
    "home_coach_win_pct_prior",
    "away_coach_win_pct_prior",
    "home_coach_win_pct_roll3",
    "away_coach_win_pct_roll3",
    "is_playoff",
    "is_final_week",
    "home_qb_switch",
    "away_qb_switch",
    "is_home_qb_new",
    "is_away_qb_new",
]

# Top-35 subset, in the order produced by the ablation study
# (descending importance: combined XGB gain + Ridge |coef| + LGB gain).
# Do NOT reorder — column order determines X_tr column order, which
# determines model coefficients. Same set, different order ⇒ different pkl.
PROD_FEATURES_35 = [
    "spread_line",
    "sack_diff",
    "sack_diff_reverse",
    "scoring_diff",
    "home_coach_win_pct_roll3",
    "scoring_diff_reverse",
    "home_coach_win_pct_prior",
    "away_rolling_allowed_play_count",
    "away_rolling_allowed_avg_yards",
    "home_rolling_win_pct",
    "diff_active_allpro_weighted",
    "epa_home_off_away_def_rolling_diff",
    "diff_active_allpro_prev_year",
    "away_season_sos_opponent_avg",
    "diff_allpro_last_3_years_weighted",
    "allpro_diff_home_off_away_def_3_years",
    "home_defense_allpro_prev_year",
    "away_offense_allpro_prev_year",
    "season_sos_diff",
    "third_down_diff",
    "epa_home_def_away_off_rolling_diff",
    "is_playoff",
    "play_count_home_def_away_off_rolling_diff",
    "home_allpro_last_3_years_weighted",
    "away_rolling_avg_yards",
    "home_rolling_allowed_avg_epa",
    "away_defense_allpro_3_years",
    "away_coach_win_pct_prior",
    "home_recent_sos_opponent_avg",
    "away_pr_prev_year",
    "away_rolling_avg_epa",
    "away_injured_count",
    "home_rolling_play_count",
    "avg_yards_home_off_away_def_rolling_diff",
    "allpro_diff_home_off_away_def_prev_year",
]


In [ ]:
if RUN_TESTS:
    import hashlib as _hashlib
    assert len(FEATURE_COLS_85) == 85, f"FEATURE_COLS_85 has {len(FEATURE_COLS_85)} entries"
    assert len(set(FEATURE_COLS_85)) == 85, "FEATURE_COLS_85 has duplicates"
    assert len(PROD_FEATURES_35) == 35, f"PROD_FEATURES_35 has {len(PROD_FEATURES_35)} entries"
    assert len(set(PROD_FEATURES_35)) == 35, "PROD_FEATURES_35 has duplicates"
    _missing = [f for f in PROD_FEATURES_35 if f not in FEATURE_COLS_85]
    assert not _missing, f"PROD_FEATURES_35 not a subset of FEATURE_COLS_85: {_missing}"
    assert "allpro_diff_home_def_away_off_3_years " in FEATURE_COLS_85, \
        "Trailing-space feature name was stripped — would break production pkl"

    # Order-preservation hash check. List order is contract: PROD_FEATURES_35 order
    # determines X_tr column order which determines pkl bytes. A reorder bug was hit
    # during Phase 2a (see memory [[feature-list-order-is-contract]]). These hashes
    # lock the canonical ablation-study order. If you intentionally change a list,
    # recompute the hash and update it here in the same commit as the retrain.
    _EXPECTED_FC85_HASH = "c1822ba82502f1963f6bde5b34270c54"
    _EXPECTED_PF35_HASH = "ac8801072e32165daadeaf83eba758e4"
    _fc85_hash = _hashlib.md5("\n".join(FEATURE_COLS_85).encode("utf-8")).hexdigest()
    _pf35_hash = _hashlib.md5("\n".join(PROD_FEATURES_35).encode("utf-8")).hexdigest()
    assert _fc85_hash == _EXPECTED_FC85_HASH, (
        f"FEATURE_COLS_85 order changed (got {_fc85_hash}, expected {_EXPECTED_FC85_HASH}). "
        "If intentional, retrain pkls and update the expected hash."
    )
    assert _pf35_hash == _EXPECTED_PF35_HASH, (
        f"PROD_FEATURES_35 order changed (got {_pf35_hash}, expected {_EXPECTED_PF35_HASH}). "
        "If intentional, retrain pkls and update the expected hash."
    )
    print(f"✓ Constants: 85 features, 35-subset, trailing-space, order-hashes — all locked.")


## Helper — `norm_name`

Normalises a player name (strips Jr./Sr./roman-numeral suffixes, accents,
punctuation; lowercases; collapses whitespace). Used to join the AllPro CSV
(`P. Mahomes`) with the injury report (`Patrick Mahomes Jr.`).

**Inputs:** any string, including non-strings (returns `""`).
**Output:** normalised lowercase string.
**Test:** asserts suffix-stripping, accent-removal, punctuation-removal,
non-string safety.


In [ ]:
def norm_name(s):
    """Strip suffixes (Jr/Sr/II/III/IV/V), punctuation, accents; lowercase."""
    if not isinstance(s, str):
        return ""
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = s.lower().strip()
    s = re.sub(r"\s+(jr\.?|sr\.?|ii|iii|iv|v)\s*$", "", s)
    s = re.sub(r"[\'.\-]", "", s)
    return re.sub(r"\s+", " ", s).strip()


# Underscore-prefixed alias retained for any code that referenced the old name.
_norm_name = norm_name


In [ ]:
if RUN_TESTS:
    assert norm_name("Patrick Mahomes Jr.")    == "patrick mahomes", "Jr. not stripped"
    assert norm_name("J.J. Watt")               == "jj watt",         "Punctuation not stripped"
    assert norm_name("Ja'Marr Chase")           == "jamarr chase",    "Apostrophe not stripped"
    assert norm_name("Odell Beckham III")       == "odell beckham",   "Roman numeral not stripped"
    assert norm_name("D'Andre Swift")           == "dandre swift",    "Leading apostrophe not stripped"
    assert norm_name(None)                       == "",                "None should yield empty string"
    assert norm_name(123)                        == "",                "Non-string should yield empty string"
    assert norm_name("  Tom  Brady  ")          == "tom brady",       "Whitespace not collapsed"
    print("✓ norm_name: 7 cases pass.")


## Helper — `canonicalize_ngs_team`

NFL Next Gen Stats labels every season with the **modern** team abbreviation
(`LAR`, `LV`, `LAC`) even for years when the team played under a different
abbreviation. This helper maps NGS abbrevs back to the schedule's per-season
abbreviation so the team merge in `_build_passer_rating` succeeds.

**Inputs:** `team_abbr` (str), `season` (int).
**Output:** the schedule-convention abbrev for that team in that season.
**Test:** asserts the three known relocations + a no-op passthrough.


In [ ]:
def canonicalize_ngs_team(team_abbr, season):
    """NGS uses modern abbreviations (LAR/LV/LAC) for historical seasons;
    map them back to the schedule's per-season convention."""
    if team_abbr == "LAR":
        return "LA"
    if team_abbr == "LV":
        return "OAK" if season < 2020 else "LV"
    if team_abbr == "LAC":
        return "SD" if season < 2017 else "LAC"
    return team_abbr


In [ ]:
if RUN_TESTS:
    # Rams relocated 2016 → "LA" everywhere in schedule
    assert canonicalize_ngs_team("LAR", 2019) == "LA"
    assert canonicalize_ngs_team("LAR", 2024) == "LA"
    # Raiders moved 2020; pre-2020 schedule uses "OAK"
    assert canonicalize_ngs_team("LV",  2018) == "OAK"
    assert canonicalize_ngs_team("LV",  2022) == "LV"
    # Chargers moved 2017; pre-2017 schedule uses "SD"
    assert canonicalize_ngs_team("LAC", 2016) == "SD"
    assert canonicalize_ngs_team("LAC", 2018) == "LAC"
    # Passthrough for unaffected teams
    assert canonicalize_ngs_team("KC",  2020) == "KC"
    print("✓ canonicalize_ngs_team: 7 cases pass.")


## Synthetic fixtures

Small synthetic schedule / PBP / AllPro tables shared across every per-group
feature test below. Keeping these in one cell means the test cells stay small
and focused on the helper they're exercising.

Layout:
* **`_synth_schedule()`** — 4 played REG-season weeks + 1 unplayed target week
  (KC @ home vs BUF, KC wins by 7 each time). Includes coaches, QBs, roof,
  surface, rest, div_game so any feature group has the inputs it needs.
* **`_synth_pbp()`** — pass plays for both teams across the prior season
  (2024, all 18 weeks) and the played weeks of the target season (2025 W1-4).
  Each game has 25 plays per team, which exceeds the >=100-attempt filter the
  passer-rating helper applies.
* **`_synth_allpro()`** — two AllPro rows so the weighted-3-year code has data.


In [ ]:
def _synth_schedule():
    """4 played weeks + 1 unplayed target week, KC vs BUF (KC wins by 7)."""
    played = [
        {"season": 2025, "week": w, "game_id": f"2025_W{w}_1", "game_type": "REG",
         "home_team": "KC", "away_team": "BUF",
         "spread_line": -3.0, "total_line": 47.0,
         "result": 7.0, "home_score": 24, "away_score": 17,
         "roof": "dome", "surface": "turf",
         "home_rest": 7, "away_rest": 7, "div_game": 0,
         "home_coach": "Andy Reid", "away_coach": "Sean McDermott",
         "home_qb_name": "P.Mahomes", "away_qb_name": "J.Allen",
         "gameday": "2025-10-05"}
        for w in range(1, 5)
    ]
    upcoming = {**played[0], "week": 5, "game_id": "2025_W5_1",
                "result": None, "home_score": None, "away_score": None}
    return pd.DataFrame(played + [upcoming])


def _synth_pbp():
    """Pass plays across 2024 (full season) and 2025 W1-4."""
    rows = [
        {"season": s, "week": w, "game_id": f"{s}_W{w}_1",
         "posteam": t, "defteam": o, "play_id": p,
         "play_type": "pass", "epa": 0.1, "yards_gained": 6,
         "sack": 0, "interception": 0, "fumble_lost": 0,
         "down": 1, "first_down": 1,
         "pass_attempt": 1, "complete_pass": 1, "passing_yards": 7, "pass_touchdown": 0,
         "passer_player_name": "P.Mahomes" if t == "KC" else "J.Allen"}
        for s in [2024, 2025]
        for w in (range(1, 19) if s == 2024 else range(1, 5))
        for t, o in [("KC", "BUF"), ("BUF", "KC")]
        for p in range(25)
    ]
    return pd.DataFrame(rows)


def _synth_allpro():
    return pd.DataFrame({
        "Year":   [2024, 2023],
        "Team":   ["KC",  "BUF"],
        "Player": ["P1",  "P2"],
        "Side":   ["offense", "defense"],
    })


## Group 1 — `_build_schedule_context`

Adds `is_playoff` and `is_final_week` boolean flags to the upcoming-week rows.
Warns if the target week contains playoff games, since the production models
were trained on REG-season only.

**Inputs:** `upcoming` (target-week games), `full_schedule`, `target_week`.
**Outputs:** `upcoming` with `is_playoff`, `is_final_week` columns.
**Test:** asserts both flag columns are present and bool-typed; final-week
detection is correct (the synth target_week=5 with REG max_week=4 → not final).


In [ ]:
def _build_schedule_context(upcoming, full_schedule, target_week):
    """Group 1: roof, surface, is_playoff, is_final_week."""
    upcoming["is_playoff"] = (upcoming["game_type"] != "REG")
    final_week_num = full_schedule[full_schedule["game_type"] == "REG"]["week"].max()
    upcoming["is_final_week"] = (
        (upcoming["game_type"] == "REG") & (upcoming["week"] == final_week_num)
    )
    if upcoming["is_playoff"].any():
        print(f"  ⚠️  Week {target_week} contains playoff games — models were trained on REG season only")
    return upcoming


In [ ]:
if RUN_TESTS:
    _fs = _synth_schedule()
    _up = _fs[_fs["week"] == 5].copy()
    _out = _build_schedule_context(_up, _fs, target_week=5)
    assert "is_playoff" in _out.columns and "is_final_week" in _out.columns
    assert bool(_out["is_playoff"].iloc[0]) is False, "REG game flagged as playoff"
    # synth has REG weeks 1-5, so max REG week = 5 → final
    assert bool(_out["is_final_week"].iloc[0]) is True, "Last REG week not flagged"
    print("✓ Group 1 — schedule_context: playoff + final-week flags correct.")


## Group 2 — `_build_rolling_pbp`

Adds 5-game rolling EPA, yards/play, and play-count for each team, both
offensive (`home/away_rolling_avg_epa`, ...) and defensive
(`home/away_rolling_allowed_avg_epa`, ...). Then computes 6 cross-matchup
diffs (off-vs-opponent-def + def-vs-opponent-off, for each of EPA / yards /
play count).

**Inputs:** `upcoming`, `pbp_s` (filtered PBP), `wk_lookup` (game_id → week/season).
**Outputs:** `upcoming` with 12 rolling cols + 6 diff cols.
**Test:** asserts all 18 expected columns exist and the diff columns equal
`home_X - away_Y` (i.e. the diff math wasn't reversed).


In [ ]:
def _build_rolling_pbp(upcoming, pbp_s, wk_lookup):
    """Group 2: rolling EPA, yards/play, play count (5-game windows)."""
    off_stats = (
        pbp_s.groupby(["game_id", "posteam"])
        .agg(avg_epa=("epa", "mean"), avg_yards=("yards_gained", "mean"), play_count=("play_id", "count"))
        .reset_index().rename(columns={"posteam": "team"})
    )
    off_stats = off_stats.merge(wk_lookup[["game_id", "week", "season"]], on="game_id", how="left")
    off_stats = off_stats.sort_values(["team", "season", "week"])
    for feat in ["avg_epa", "avg_yards", "play_count"]:
        off_stats[f"rolling_{feat}"] = (
            off_stats.groupby("team")[feat]
            .transform(lambda x: x.rolling(5, min_periods=1).mean())
        )

    def_stats = (
        pbp_s.groupby(["game_id", "defteam"])
        .agg(allowed_avg_epa=("epa", "mean"), allowed_avg_yards=("yards_gained", "mean"), allowed_play_count=("play_id", "count"))
        .reset_index().rename(columns={"defteam": "team"})
    )
    def_stats = def_stats.merge(wk_lookup[["game_id", "week", "season"]], on="game_id", how="left")
    def_stats = def_stats.sort_values(["team", "season", "week"])
    for feat in ["allowed_avg_epa", "allowed_avg_yards", "allowed_play_count"]:
        def_stats[f"rolling_{feat}"] = (
            def_stats.groupby("team")[feat]
            .transform(lambda x: x.rolling(5, min_periods=1).mean())
        )

    latest_off = off_stats.groupby("team").nth(-1).reset_index()
    latest_def = def_stats.groupby("team").nth(-1).reset_index()

    for side, df in [("home_team", latest_off), ("away_team", latest_off)]:
        prefix = "home_" if side == "home_team" else "away_"
        cols = [c for c in df.columns if c.startswith("rolling_")]
        upcoming = upcoming.merge(
            df[["team"] + cols].rename(columns={"team": side, **{c: f"{prefix}{c}" for c in cols}}),
            on=side, how="left"
        )
    for side, df in [("home_team", latest_def), ("away_team", latest_def)]:
        prefix = "home_" if side == "home_team" else "away_"
        cols = [c for c in df.columns if c.startswith("rolling_")]
        upcoming = upcoming.merge(
            df[["team"] + cols].rename(columns={"team": side, **{c: f"{prefix}{c}" for c in cols}}),
            on=side, how="left"
        )

    upcoming["epa_home_off_away_def_rolling_diff"]        = upcoming["home_rolling_avg_epa"]            - upcoming["away_rolling_allowed_avg_epa"]
    upcoming["epa_home_def_away_off_rolling_diff"]        = upcoming["home_rolling_allowed_avg_epa"]    - upcoming["away_rolling_avg_epa"]
    upcoming["avg_yards_home_off_away_def_rolling_diff"]  = upcoming["home_rolling_avg_yards"]          - upcoming["away_rolling_allowed_avg_yards"]
    upcoming["avg_yards_home_def_away_off_rolling_diff"]  = upcoming["home_rolling_allowed_avg_yards"]  - upcoming["away_rolling_avg_yards"]
    upcoming["play_count_home_off_away_def_rolling_diff"] = upcoming["home_rolling_play_count"]         - upcoming["away_rolling_allowed_play_count"]
    upcoming["play_count_home_def_away_off_rolling_diff"] = upcoming["home_rolling_allowed_play_count"] - upcoming["away_rolling_play_count"]
    return upcoming


In [ ]:
if RUN_TESTS:
    _fs = _synth_schedule()
    _pbp = _synth_pbp()
    _up = _fs[_fs["week"] == 5].copy()
    _wk = _pbp[["game_id", "week", "season"]].drop_duplicates()
    _out = _build_rolling_pbp(_up, _pbp, _wk)
    _expected = [
        "home_rolling_avg_epa", "home_rolling_avg_yards", "home_rolling_play_count",
        "away_rolling_avg_epa", "away_rolling_avg_yards", "away_rolling_play_count",
        "home_rolling_allowed_avg_epa", "home_rolling_allowed_avg_yards", "home_rolling_allowed_play_count",
        "away_rolling_allowed_avg_epa", "away_rolling_allowed_avg_yards", "away_rolling_allowed_play_count",
        "epa_home_off_away_def_rolling_diff", "epa_home_def_away_off_rolling_diff",
        "avg_yards_home_off_away_def_rolling_diff", "avg_yards_home_def_away_off_rolling_diff",
        "play_count_home_off_away_def_rolling_diff", "play_count_home_def_away_off_rolling_diff",
    ]
    for c in _expected:
        assert c in _out.columns, f"Missing column {c}"
    # Diff math sanity: epa_home_off_away_def_rolling_diff == home_rolling_avg_epa - away_rolling_allowed_avg_epa
    r = _out.iloc[0]
    assert np.isclose(r["epa_home_off_away_def_rolling_diff"],
                      r["home_rolling_avg_epa"] - r["away_rolling_allowed_avg_epa"]), \
        "epa diff math is wrong"
    print(f"✓ Group 2 — rolling_pbp: {len(_expected)} cols present, diff math correct.")


## Groups 3 & 5 — `_build_sos_and_performance`

Combined because Group 5 builds on the long-format team-game DataFrame Group 3
constructs. Adds:

* **SOS** — `home/away_recent_sos_opponent_avg` (rolling-3 of opponent win%),
  `home/away_season_sos_opponent_avg` (expanding mean), `sos_diff`, `season_sos_diff`
* **Rolling win%** — `home/away_rolling_win_pct` (5-game window)
* **Scoring** — `home/away_rolling_scored`, `home/away_rolling_allowed`, `scoring_diff`, `scoring_diff_reverse`
* **Cover rate** — `home/away_rolling_cover_rate`, `cover_rate_diff`
* **League margin** — `league_rolling_avg_abs_margin_by_week`

**Test:** asserts the headline 6 columns exist and are non-NaN for the synthetic
input (KC won every game → home_rolling_win_pct == 1.0).


In [ ]:
def _build_sos_and_performance(upcoming, _hist_rolling, history, week_margin_lkp):
    """Groups 3+5: SOS, rolling win%, scoring, cover rate, league margin.

    Combined because Group 5 builds on the long_df constructed in Group 3.
    """
    # ── Build shared long-format team-game DataFrame ──────────────────────────
    home_g = _hist_rolling[["season", "week", "home_team", "away_team", "home_score", "away_score"]].copy()
    home_g.columns = ["season", "week", "team", "opponent", "team_score", "opp_score"]
    away_g = _hist_rolling[["season", "week", "away_team", "home_team", "away_score", "home_score"]].copy()
    away_g.columns = ["season", "week", "team", "opponent", "team_score", "opp_score"]
    long_df = pd.concat([home_g, away_g]).sort_values(["team", "season", "week"])
    long_df["team_win"] = (long_df["team_score"] > long_df["opp_score"]).astype(int)

    # ── Group 3 — SOS ────────────────────────────────────────────────────────
    long_df["win_pct"] = long_df.groupby("team")["team_win"].transform(
        lambda x: x.shift(1).expanding().mean()
    )
    opp_wp = long_df[["season", "week", "team", "win_pct"]].copy()
    opp_wp.columns = ["season", "week", "opponent", "opponent_win_pct"]
    long_df = long_df.merge(opp_wp, on=["season", "week", "opponent"], how="left")
    long_df["recent_sos"] = long_df.groupby("team")["opponent_win_pct"].transform(
        lambda x: x.rolling(3, min_periods=1).mean().fillna(0)
    )
    # Accumulates across seasons in long_df (target_season-1 + current games) — name matches
    # production model feature exactly; do not rename without retraining.
    long_df["season_sos"] = long_df.groupby("team")["opponent_win_pct"].transform(
        lambda x: x.expanding().mean().fillna(0)
    )
    latest_sos = long_df.groupby("team").nth(-1).reset_index()[["team", "recent_sos", "season_sos"]]
    upcoming = upcoming.merge(latest_sos.rename(columns={"team": "home_team", "recent_sos": "home_recent_sos_opponent_avg", "season_sos": "home_season_sos_opponent_avg"}), on="home_team", how="left")
    upcoming = upcoming.merge(latest_sos.rename(columns={"team": "away_team", "recent_sos": "away_recent_sos_opponent_avg", "season_sos": "away_season_sos_opponent_avg"}), on="away_team", how="left")
    for col in ["home_recent_sos_opponent_avg", "home_season_sos_opponent_avg", "away_recent_sos_opponent_avg", "away_season_sos_opponent_avg"]:
        upcoming[col] = upcoming[col].fillna(0)
    upcoming["sos_diff"]        = upcoming["home_recent_sos_opponent_avg"] - upcoming["away_recent_sos_opponent_avg"]
    upcoming["season_sos_diff"] = upcoming["home_season_sos_opponent_avg"] - upcoming["away_season_sos_opponent_avg"]

    # ── Group 5 — rolling win%, scoring, cover rate, league margin ────────────
    long_df["rolling_win_pct"] = long_df.groupby("team")["team_win"].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )
    latest_wp = long_df.groupby("team").nth(-1).reset_index()[["team", "rolling_win_pct"]]
    upcoming = upcoming.merge(latest_wp.rename(columns={"team": "home_team", "rolling_win_pct": "home_rolling_win_pct"}), on="home_team", how="left")
    upcoming = upcoming.merge(latest_wp.rename(columns={"team": "away_team", "rolling_win_pct": "away_rolling_win_pct"}), on="away_team", how="left")

    long_df["rolling_scored"]  = long_df.groupby("team")["team_score"].transform(lambda x: x.rolling(5, min_periods=1).mean())
    long_df["rolling_allowed"] = long_df.groupby("team")["opp_score"].transform(lambda x: x.rolling(5, min_periods=1).mean())
    latest_sc = long_df.groupby("team").nth(-1).reset_index()[["team", "rolling_scored", "rolling_allowed"]]
    upcoming = upcoming.merge(latest_sc.rename(columns={"team": "home_team", "rolling_scored": "home_rolling_scored", "rolling_allowed": "home_rolling_allowed"}), on="home_team", how="left")
    upcoming = upcoming.merge(latest_sc.rename(columns={"team": "away_team", "rolling_scored": "away_rolling_scored", "rolling_allowed": "away_rolling_allowed"}), on="away_team", how="left")
    upcoming["scoring_diff"]         = upcoming["home_rolling_scored"] - upcoming["away_rolling_scored"]
    upcoming["scoring_diff_reverse"] = upcoming["away_rolling_scored"] - upcoming["home_rolling_scored"]

    history2 = _hist_rolling.copy()
    # push=0 (home) / push=1 (away) matches training behaviour — retrain before switching to NaN
    history2["home_covered"] = (history2["result"] > history2["spread_line"]).astype(int)
    home_cov = history2[["season", "week", "home_team", "home_covered"]].rename(columns={"home_team": "team", "home_covered": "covered"})
    away_cov = history2[["season", "week", "away_team", "home_covered"]].copy()
    away_cov["covered"] = 1 - away_cov["home_covered"]
    away_cov = away_cov.drop(columns="home_covered").rename(columns={"away_team": "team"})
    cover_df = pd.concat([home_cov, away_cov]).sort_values(["team", "season", "week"])
    cover_df["rolling_cover_rate"] = cover_df.groupby("team")["covered"].transform(lambda x: x.rolling(5, min_periods=1).mean())
    latest_cover = cover_df.groupby("team").nth(-1).reset_index()[["team", "rolling_cover_rate"]]
    upcoming = upcoming.merge(latest_cover.rename(columns={"team": "home_team", "rolling_cover_rate": "home_rolling_cover_rate"}), on="home_team", how="left")
    upcoming = upcoming.merge(latest_cover.rename(columns={"team": "away_team", "rolling_cover_rate": "away_rolling_cover_rate"}), on="away_team", how="left")
    upcoming["cover_rate_diff"] = upcoming["home_rolling_cover_rate"] - upcoming["away_rolling_cover_rate"]

    # Cross-season per-week avg absolute margin — matches training definition
    _target_week = int(upcoming["week"].iloc[0])
    if week_margin_lkp is not None and _target_week in week_margin_lkp.index:
        upcoming["league_rolling_avg_abs_margin_by_week"] = float(week_margin_lkp[_target_week])
    elif week_margin_lkp is not None and len(week_margin_lkp) > 0:
        upcoming["league_rolling_avg_abs_margin_by_week"] = float(week_margin_lkp.mean())
    else:
        _wk_avgs = history.groupby("week")["result"].apply(lambda x: x.abs().mean()).sort_index()
        upcoming["league_rolling_avg_abs_margin_by_week"] = float(_wk_avgs.iloc[-1]) if len(_wk_avgs) > 0 else 0.0

    return upcoming


In [ ]:
if RUN_TESTS:
    _fs = _synth_schedule()
    _up = _fs[_fs["week"] == 5].copy()
    _history = _fs[(_fs["week"] < 5) & _fs["result"].notna()].copy()
    _hist_rolling = _history[["season","week","home_team","away_team","home_score","away_score","result","spread_line"]].copy()
    _out = _build_sos_and_performance(_up, _hist_rolling, _history, week_margin_lkp=None)
    for c in ["sos_diff", "season_sos_diff", "home_rolling_win_pct", "away_rolling_win_pct",
              "scoring_diff", "cover_rate_diff", "league_rolling_avg_abs_margin_by_week"]:
        assert c in _out.columns, f"Missing column {c}"
        assert not pd.isna(_out[c].iloc[0]), f"Column {c} is NaN"
    # KC won every game in synth → home_rolling_win_pct should be 1.0
    assert _out["home_rolling_win_pct"].iloc[0] == 1.0, \
        f"KC won 4/4 synth games — expected 1.0, got {_out['home_rolling_win_pct'].iloc[0]}"
    assert _out["away_rolling_win_pct"].iloc[0] == 0.0, "BUF lost 4/4 synth games — expected 0.0"
    print("✓ Groups 3+5 — sos_and_performance: KC win%=1.0, BUF win%=0.0, all cols present.")


## Group 4 — `_build_allpro`

Weighted 3-year AllPro roster quality, split by offense and defense. The
weighting is 4 / 2 / 1 for the most-recent / 2-years-ago / 3-years-ago AllPro
nominations. Plus prev-year counts (overall / offense / defense). Then computes
6 diff columns between home and away.

**Inputs:** `upcoming`, `allpro_df`, `target_season`.
**Outputs:** `upcoming` with 12 weighted/count columns + 6 diff columns.
**Test:** asserts all 18 columns present and the trailing-space column name is
preserved exactly.


In [ ]:
def _build_allpro(upcoming, allpro_df, target_season):
    """Group 4: weighted 3-year AllPro roster quality (offense/defense split)."""
    offense_df = allpro_df[allpro_df["Side"] == "offense"].copy()
    defense_df = allpro_df[allpro_df["Side"] == "defense"].copy()

    def build_weighted(df_ap):
        frames = []
        for year in range(2006, target_season + 1):
            curr = []
            for yrs_back, weight in zip([1, 2, 3], [4, 2, 1]):
                tmp = df_ap[df_ap["Year"] == year - yrs_back].copy()
                tmp["Weight"] = weight
                tmp["Year_target"] = year
                curr.append(tmp)
            comb    = pd.concat(curr)
            deduped = comb.sort_values("Weight", ascending=False).drop_duplicates(["Player", "Year_target"])
            wc      = deduped.groupby(["Year_target", "Team"])["Weight"].sum().reset_index()
            wc.columns = ["season", "Team", "allpro_weighted"]
            frames.append(wc)
        return pd.concat(frames, ignore_index=True)

    weighted_allpro  = build_weighted(allpro_df)
    offense_weighted = build_weighted(offense_df)
    defense_weighted = build_weighted(defense_df)

    def merge_allpro(df, feat_df, feat_col, home_col, away_col):
        lookup = feat_df[feat_df["season"] == target_season].drop(columns="season")
        df = df.merge(lookup.rename(columns={"Team": "home_team", feat_col: home_col}), on="home_team", how="left")
        df = df.merge(lookup.rename(columns={"Team": "away_team", feat_col: away_col}), on="away_team", how="left")
        df[home_col] = df[home_col].fillna(0)
        df[away_col] = df[away_col].fillna(0)
        return df

    upcoming = merge_allpro(upcoming, weighted_allpro,  "allpro_weighted", "home_allpro_last_3_years_weighted", "away_allpro_last_3_years_weighted")
    upcoming = merge_allpro(upcoming, offense_weighted, "allpro_weighted", "home_offense_allpro_3_years",        "away_offense_allpro_3_years")
    upcoming = merge_allpro(upcoming, defense_weighted, "allpro_weighted", "home_defense_allpro_3_years",        "away_defense_allpro_3_years")

    prev_overall = allpro_df.assign(season=allpro_df["Year"] + 1).groupby(["season", "Team"])["Player"].nunique().reset_index(name="allpro_prev_year")
    prev_offense = offense_df.assign(season=offense_df["Year"] + 1).groupby(["season", "Team"])["Player"].nunique().reset_index(name="allpro_prev_year")
    prev_defense = defense_df.assign(season=defense_df["Year"] + 1).groupby(["season", "Team"])["Player"].nunique().reset_index(name="allpro_prev_year")

    upcoming = merge_allpro(upcoming, prev_overall, "allpro_prev_year", "home_allpro_prev_year",         "away_allpro_prev_year")
    upcoming = merge_allpro(upcoming, prev_offense, "allpro_prev_year", "home_offense_allpro_prev_year", "away_offense_allpro_prev_year")
    upcoming = merge_allpro(upcoming, prev_defense, "allpro_prev_year", "home_defense_allpro_prev_year", "away_defense_allpro_prev_year")

    upcoming["diff_allpro_last_3_years_weighted"]        = upcoming["home_allpro_last_3_years_weighted"] - upcoming["away_allpro_last_3_years_weighted"]
    upcoming["diff_allpro_prev_year"]                    = upcoming["home_allpro_prev_year"]              - upcoming["away_allpro_prev_year"]
    upcoming["allpro_diff_home_off_away_def_3_years"]    = upcoming["home_offense_allpro_3_years"]        - upcoming["away_defense_allpro_3_years"]
    upcoming["allpro_diff_home_def_away_off_3_years "]   = upcoming["home_defense_allpro_3_years"]        - upcoming["away_offense_allpro_3_years"]   # trailing space matches model
    upcoming["allpro_diff_home_off_away_def_prev_year"]  = upcoming["home_offense_allpro_prev_year"]      - upcoming["away_defense_allpro_prev_year"]
    upcoming["allpro_diff_home_def_away_off_prev_year"]  = upcoming["home_defense_allpro_prev_year"]      - upcoming["away_offense_allpro_prev_year"]
    return upcoming


In [ ]:
if RUN_TESTS:
    _fs = _synth_schedule()
    _up = _fs[_fs["week"] == 5].copy()
    _allpro = _synth_allpro()
    _out = _build_allpro(_up, _allpro, target_season=2025)
    for c in ["home_allpro_last_3_years_weighted", "away_allpro_last_3_years_weighted",
              "home_offense_allpro_3_years", "away_offense_allpro_3_years",
              "home_defense_allpro_3_years", "away_defense_allpro_3_years",
              "diff_allpro_last_3_years_weighted",
              "allpro_diff_home_def_away_off_3_years "]:  # trailing space!
        assert c in _out.columns, f"Missing column {c}"
    # KC has an offense AllPro in 2024 (weight 4 for target_season 2025) → home_offense_allpro_3_years = 4
    assert _out["home_offense_allpro_3_years"].iloc[0] == 4.0, \
        f"Expected KC offense_3yr=4, got {_out['home_offense_allpro_3_years'].iloc[0]}"
    print("✓ Group 4 — allpro: trailing-space col preserved, weighted math correct.")


## Group 6 — `_build_situational_pbp`

5-game rolling sacks, turnovers, and third-down conversion rate per team, then
6 diff columns (forward and reverse) for sack / turnover / 3rd-down.

**Inputs:** `upcoming`, `pbp_s`, `wk_lookup`.
**Outputs:** 6 diff columns plus the underlying `home/away_rolling_*` columns.
**Test:** asserts diff columns exist and the diff_reverse equals -diff.


In [ ]:
def _build_situational_pbp(upcoming, pbp_s, wk_lookup):
    """Group 6: sacks, turnovers, third-down conversion rate (5-game rolling)."""
    sack_df = pbp_s[pbp_s["sack"] == 1].copy()
    sacks   = sack_df.groupby(["game_id", "defteam"]).size().reset_index(name="sacks").rename(columns={"defteam": "team"})
    sacks   = sacks.merge(wk_lookup[["game_id", "week", "season"]], on="game_id", how="left").sort_values(["team", "season", "week"])
    sacks["rolling_sacks"] = sacks.groupby("team")["sacks"].transform(lambda x: x.rolling(5, min_periods=1).mean())
    latest_sacks = sacks.groupby("team").nth(-1).reset_index()[["team", "rolling_sacks"]]
    upcoming = upcoming.merge(latest_sacks.rename(columns={"team": "home_team", "rolling_sacks": "home_rolling_sacks"}), on="home_team", how="left")
    upcoming = upcoming.merge(latest_sacks.rename(columns={"team": "away_team", "rolling_sacks": "away_rolling_sacks"}), on="away_team", how="left")
    upcoming["sack_diff"]         = upcoming["home_rolling_sacks"] - upcoming["away_rolling_sacks"]
    upcoming["sack_diff_reverse"] = upcoming["away_rolling_sacks"] - upcoming["home_rolling_sacks"]

    pbp_s2 = pbp_s.copy()
    pbp_s2["turnover"] = ((pbp_s2["interception"] == 1) | (pbp_s2["fumble_lost"] == 1)).astype(int)
    to_df = pbp_s2.groupby(["game_id", "posteam"])["turnover"].sum().reset_index().rename(columns={"posteam": "team"})
    to_df = to_df.merge(wk_lookup[["game_id", "week", "season"]], on="game_id", how="left").sort_values(["team", "season", "week"])
    to_df["rolling_turnovers"] = to_df.groupby("team")["turnover"].transform(lambda x: x.rolling(5, min_periods=1).mean())
    latest_to = to_df.groupby("team").nth(-1).reset_index()[["team", "rolling_turnovers"]]
    upcoming  = upcoming.merge(latest_to.rename(columns={"team": "home_team", "rolling_turnovers": "home_rolling_turnovers"}), on="home_team", how="left")
    upcoming  = upcoming.merge(latest_to.rename(columns={"team": "away_team", "rolling_turnovers": "away_rolling_turnovers"}), on="away_team", how="left")
    upcoming["turnover_diff"]         = upcoming["home_rolling_turnovers"] - upcoming["away_rolling_turnovers"]
    upcoming["turnover_diff_reverse"] = upcoming["away_rolling_turnovers"] - upcoming["home_rolling_turnovers"]

    pbp_s2["third_att"]  = (pbp_s2["down"] == 3).astype(int)
    pbp_s2["third_conv"] = ((pbp_s2["down"] == 3) & (pbp_s2["first_down"] == 1)).astype(int)
    third_df = pbp_s2.groupby(["game_id", "posteam"]).agg(third_att=("third_att", "sum"), third_conv=("third_conv", "sum")).reset_index().rename(columns={"posteam": "team"})
    third_df["third_down_rate"] = third_df["third_conv"] / third_df["third_att"].replace(0, 1)
    third_df = third_df.merge(wk_lookup[["game_id", "week", "season"]], on="game_id", how="left").sort_values(["team", "season", "week"])
    third_df["rolling_third"] = third_df.groupby("team")["third_down_rate"].transform(lambda x: x.rolling(5, min_periods=1).mean())
    latest_third = third_df.groupby("team").nth(-1).reset_index()[["team", "rolling_third"]]
    upcoming = upcoming.merge(latest_third.rename(columns={"team": "home_team", "rolling_third": "home_rolling_third_down"}), on="home_team", how="left")
    upcoming = upcoming.merge(latest_third.rename(columns={"team": "away_team", "rolling_third": "away_rolling_third_down"}), on="away_team", how="left")
    upcoming["third_down_diff"]         = upcoming["home_rolling_third_down"] - upcoming["away_rolling_third_down"]
    upcoming["third_down_diff_reverse"] = upcoming["away_rolling_third_down"] - upcoming["home_rolling_third_down"]
    return upcoming


In [ ]:
if RUN_TESTS:
    _fs = _synth_schedule()
    _pbp = _synth_pbp()
    _up = _fs[_fs["week"] == 5].copy()
    _wk = _pbp[["game_id", "week", "season"]].drop_duplicates()
    _out = _build_situational_pbp(_up, _pbp, _wk)
    for c in ["turnover_diff", "turnover_diff_reverse",
              "third_down_diff", "third_down_diff_reverse"]:
        assert c in _out.columns, f"Missing column {c}"
    # sack_diff may be NaN since synth pbp has no sacks for either team (sack=0 everywhere)
    # — but the columns must exist.
    assert "sack_diff" in _out.columns and "sack_diff_reverse" in _out.columns
    # reverse should equal -forward
    r = _out.iloc[0]
    assert np.isclose(r["turnover_diff_reverse"], -r["turnover_diff"]), "turnover_diff_reverse != -turnover_diff"
    assert np.isclose(r["third_down_diff_reverse"], -r["third_down_diff"]), "third_down_diff_reverse != -third_down_diff"
    print("✓ Group 6 — situational_pbp: diffs + reverse-diff invariant hold.")


## Group 7 — `_build_qb_switch`

Flags whether each team's QB this week is different from the QB they ended the
prior game with (`home/away_qb_switch`) and a parallel `is_home/away_qb_new`
flag (currently the same — kept distinct so the two pairs could diverge in a
future retrain).

**Inputs:** `upcoming`, `history`, optional `coach_hist_df` (full schedule with
prior-season results), `target_season`.
**Output:** 4 boolean columns.
**Test:** asserts both flag pairs are False when the synth QB never changes.


In [ ]:
def _build_qb_switch(upcoming, history, coach_hist_df, target_season):
    """Group 7: QB switch flags (home/away qb_switch and is_qb_new)."""
    if coach_hist_df is not None:
        _prior_sched = coach_hist_df[
            (coach_hist_df["season"] == target_season - 1) & coach_hist_df["result"].notna()
        ].copy()
    else:
        try:
            _prior_raw   = nfl.load_schedules([target_season - 1])
            _prior_sched = _prior_raw.to_pandas() if hasattr(_prior_raw, "to_pandas") else pd.DataFrame(_prior_raw)
            _prior_sched = _prior_sched[_prior_sched["result"].notna()].copy()
        except Exception:
            _prior_sched = pd.DataFrame(columns=list(history.columns))
    _qb_hist = pd.concat([
        _prior_sched[["season", "week", "home_team", "away_team", "home_qb_name", "away_qb_name"]],
        history[["season", "week", "home_team", "away_team", "home_qb_name", "away_qb_name"]]
    ]).dropna(subset=["home_team", "away_team"])
    home_qbs = _qb_hist[["season", "week", "home_team", "home_qb_name"]].rename(columns={"home_team": "team", "home_qb_name": "qb_name"})
    away_qbs = _qb_hist[["season", "week", "away_team", "away_qb_name"]].rename(columns={"away_team": "team", "away_qb_name": "qb_name"})
    team_qbs = pd.concat([home_qbs, away_qbs]).sort_values(["team", "season", "week"])
    last_qb  = team_qbs.groupby("team").nth(-1).reset_index()[["team", "qb_name"]].rename(columns={"qb_name": "last_qb"})
    upcoming = upcoming.merge(last_qb.rename(columns={"team": "home_team", "last_qb": "home_last_qb"}), on="home_team", how="left")
    upcoming = upcoming.merge(last_qb.rename(columns={"team": "away_team", "last_qb": "away_last_qb"}), on="away_team", how="left")
    upcoming["home_qb_switch"] = (
        upcoming["home_qb_name"].notna() &
        upcoming["home_last_qb"].notna() &
        (upcoming["home_qb_name"] != upcoming["home_last_qb"])
    )
    upcoming["away_qb_switch"] = (
        upcoming["away_qb_name"].notna() &
        upcoming["away_last_qb"].notna() &
        (upcoming["away_qb_name"] != upcoming["away_last_qb"])
    )
    # Both pairs are currently synonymous; differentiate only if models are retrained
    upcoming["is_home_qb_new"] = upcoming["home_qb_switch"]
    upcoming["is_away_qb_new"] = upcoming["away_qb_switch"]
    return upcoming


In [ ]:
if RUN_TESTS:
    _fs = _synth_schedule()
    _hist = _fs[(_fs["week"] < 5) & _fs["result"].notna()].copy()
    _up = _fs[_fs["week"] == 5].copy()
    _coach_hist = _fs[_fs["result"].notna()].copy()
    _out = _build_qb_switch(_up, _hist, _coach_hist, target_season=2025)
    for c in ["home_qb_switch", "away_qb_switch", "is_home_qb_new", "is_away_qb_new"]:
        assert c in _out.columns, f"Missing column {c}"
    # Synth has P.Mahomes / J.Allen every week → no switch
    assert bool(_out["home_qb_switch"].iloc[0]) is False, "False switch detected for P.Mahomes"
    assert bool(_out["away_qb_switch"].iloc[0]) is False, "False switch detected for J.Allen"
    print("✓ Group 7 — qb_switch: 4 flag cols present, no false switches.")


## Group 8 — `_build_passer_rating`

Prior-season passer rating + completion-% above expectation + avg time-to-throw,
sourced from NFL Next Gen Stats (2016+) with a manual-passer-rating fallback
for pre-2016 seasons (or any year where the NGS load fails). NGS abbrevs go
through `canonicalize_ngs_team` before the merge.

**Inputs:** `upcoming`, `pbp_rp`, `target_season`.
**Outputs:** 9 columns — `home/away/diff_pr_prev_year` + `..._cpae_prev_year` +
`..._time_to_throw_prev_year`.
**Test:** asserts all 9 columns exist and are non-NaN after median imputation.


In [ ]:
def _build_passer_rating(upcoming, pbp_rp, target_season, ngs_data=None):
    """Group 8: prior-season NFL passer rating diff.

    Source: NFL Next Gen Stats (`nfl.load_nextgen_stats`, 2016+) — the official
    league source. Falls back to the manual NFL-passer-rating formula on PBP
    if NGS is unavailable, or the prior season predates NGS (pre-2016).

    Args:
        upcoming, pbp_rp, target_season: as documented in build_features.
        ngs_data: optional pre-fetched DataFrame with columns ``team_abbr``,
            ``season``, ``week``, ``attempts``, ``passer_rating``,
            ``completion_percentage_above_expectation``, ``avg_time_to_throw``.
            If provided, skips the live ``nfl.load_nextgen_stats`` call — used
            by inline tests for hermetic execution.
    """
    prior_season = target_season - 1

    pr_prev = None
    if prior_season >= 2016:
        try:
            if ngs_data is not None:
                ngs = ngs_data.copy()
            else:
                ngs = nfl.load_nextgen_stats(seasons=[prior_season], stat_type="passing").to_pandas()
            ngs_agg = ngs[(ngs["week"] == 0) & (ngs["attempts"] >= 100)].copy()
            if not ngs_agg.empty:
                ngs_agg["posteam"] = ngs_agg["team_abbr"].apply(lambda t: canonicalize_ngs_team(t, prior_season))
                pr_prev = (ngs_agg.sort_values("attempts", ascending=False)
                                  .groupby("posteam").first().reset_index()
                                  [["posteam", "passer_rating",
                                    "completion_percentage_above_expectation",
                                    "avg_time_to_throw"]])
        except Exception as _e:
            print(f"  WARNING: NGS load failed for {prior_season} ({_e}) — falling back to manual passer-rating")

    if pr_prev is None or pr_prev.empty:
        # Manual NFL passer rating from PBP — pre-2016 or NGS unavailable
        pass_plays = pbp_rp[
            (pbp_rp["play_type"] == "pass") &
            (pbp_rp["season"]    == prior_season) &
            (pbp_rp["passer_player_name"].notna())
        ].copy()
        qb_stats = pass_plays.groupby(["season", "posteam", "passer_player_name"]).agg(
            attempts=("pass_attempt", "sum"), completions=("complete_pass", "sum"),
            yards=("passing_yards", "sum"), tds=("pass_touchdown", "sum"), ints=("interception", "sum")
        ).reset_index()
        qb_stats = qb_stats[qb_stats["attempts"] >= 100]

        def passer_rating(row):
            a = max(0, min(((row["completions"] / row["attempts"]) - 0.3) * 5, 2.375))
            b = max(0, min(((row["yards"]       / row["attempts"]) - 3) * 0.25, 2.375))
            c = max(0, min(row["tds"]           / row["attempts"]  * 20,        2.375))
            d = max(0, min(2.375 - (row["ints"] / row["attempts"] * 25),        2.375))
            return ((a + b + c + d) / 6) * 100

        qb_stats["passer_rating"] = qb_stats.apply(passer_rating, axis=1)
        pr_prev = (qb_stats.sort_values("attempts", ascending=False)
                          .groupby(["season", "posteam"]).first().reset_index()
                          [["posteam", "passer_rating"]])

    median_pr = pr_prev["passer_rating"].median()
    upcoming = upcoming.merge(
        pr_prev[["posteam", "passer_rating"]].rename(columns={"posteam": "home_team", "passer_rating": "home_pr_prev_year"}),
        on="home_team", how="left")
    upcoming = upcoming.merge(
        pr_prev[["posteam", "passer_rating"]].rename(columns={"posteam": "away_team", "passer_rating": "away_pr_prev_year"}),
        on="away_team", how="left")
    upcoming["home_pr_prev_year"] = upcoming["home_pr_prev_year"].fillna(median_pr)
    upcoming["away_pr_prev_year"] = upcoming["away_pr_prev_year"].fillna(median_pr)
    upcoming["diff_pr_prev_year"] = upcoming["home_pr_prev_year"] - upcoming["away_pr_prev_year"]
    for _ngs_col, _feat in [
        ("completion_percentage_above_expectation", "cpae_prev_year"),
        ("avg_time_to_throw", "time_to_throw_prev_year"),
    ]:
        if _ngs_col in pr_prev.columns and pr_prev[_ngs_col].notna().any():
            _med = pr_prev[_ngs_col].median()
            _sub = pr_prev[["posteam", _ngs_col]].dropna(subset=[_ngs_col])
            upcoming = upcoming.merge(
                _sub.rename(columns={"posteam": "home_team", _ngs_col: f"home_{_feat}"}),
                on="home_team", how="left")
            upcoming = upcoming.merge(
                _sub.rename(columns={"posteam": "away_team", _ngs_col: f"away_{_feat}"}),
                on="away_team", how="left")
        else:
            _med = 0.0
            upcoming[f"home_{_feat}"] = float("nan")
            upcoming[f"away_{_feat}"] = float("nan")
        upcoming[f"home_{_feat}"] = upcoming[f"home_{_feat}"].fillna(_med)
        upcoming[f"away_{_feat}"] = upcoming[f"away_{_feat}"].fillna(_med)
        upcoming[f"diff_{_feat}"] = upcoming[f"home_{_feat}"] - upcoming[f"away_{_feat}"]
    return upcoming


In [ ]:
if RUN_TESTS:
    _fs = _synth_schedule()
    _pbp = _synth_pbp()
    _up = _fs[_fs["week"] == 5].copy()

    # Hermetic test: pre-fab the NGS aggregate that ``_build_passer_rating`` would
    # otherwise fetch via ``nfl.load_nextgen_stats``. Mirrors NGS schema (week==0 row
    # per team with attempts >= 100, plus the three columns we read).
    _ngs_stub = pd.DataFrame([
        {"team_abbr": "KC",  "season": 2024, "week": 0, "attempts": 500,
         "passer_rating": 102.5, "completion_percentage_above_expectation": 4.5, "avg_time_to_throw": 2.7},
        {"team_abbr": "BUF", "season": 2024, "week": 0, "attempts": 480,
         "passer_rating":  88.3, "completion_percentage_above_expectation": -1.2, "avg_time_to_throw": 2.9},
    ])
    _out = _build_passer_rating(_up, _pbp, target_season=2025, ngs_data=_ngs_stub)
    for c in ["home_pr_prev_year", "away_pr_prev_year", "diff_pr_prev_year",
              "home_cpae_prev_year", "away_cpae_prev_year", "diff_cpae_prev_year",
              "home_time_to_throw_prev_year", "away_time_to_throw_prev_year", "diff_time_to_throw_prev_year"]:
        assert c in _out.columns, f"Missing column {c}"
        assert not pd.isna(_out[c].iloc[0]), f"Column {c} is NaN after median imputation"
    # KC (home) has the higher passer rating in the stub, BUF (away) is lower — diff should be positive.
    assert _out["diff_pr_prev_year"].iloc[0] > 0, "Expected positive diff (KC > BUF)"
    print(f"✓ Group 8 — passer_rating: 9 cols populated, hermetic (no network), home_pr={_out['home_pr_prev_year'].iloc[0]:.1f}")


## Group 9 — `_build_injuries`

Counts injured players (Out + Doubtful) per team for the target week, and the
AllPro-weighted injury impact (active AllPro weight = team's 3-year weight
minus the weight tied up in injured AllPros). Falls back to zero-fill on any
nflreadpy failure.

**Depends on Group 4** — reads `home/away_allpro_last_3_years_weighted` and
`home/away_allpro_prev_year` from `upcoming`.

**Inputs:** `upcoming` (with Group 4 cols), `allpro_df`, `target_season`, `target_week`.
**Outputs:** 5 columns — `home/away_injured_count`, `diff_injured_count`,
`diff_active_allpro_weighted`, `diff_active_allpro_prev_year`.
**Test:** asserts the 5 columns exist (values will be 0 in offline synth since
nflreadpy.load_injuries fails without network).


In [ ]:
def _build_injuries(upcoming, allpro_df, target_season, target_week):
    """Group 9: injured player count and AllPro-weighted injury impact.

    Reads home/away_allpro_last_3_years_weighted from upcoming, so must run
    after _build_allpro (Group 4).
    """
    try:
        raw_inj = nfl.load_injuries(seasons=[target_season])
        inj_df  = raw_inj.to_pandas() if hasattr(raw_inj, "to_pandas") else pd.DataFrame(raw_inj)
        _STATUS_WEIGHT = {"Out": 1.0, "Doubtful": 0.75}
        inj_week = inj_df[(inj_df["report_status"].isin(["Out", "Doubtful"])) & (inj_df["week"] == target_week)].copy()
        inj_by_team = inj_week.groupby("team").size().reset_index(name="count")

        upcoming = upcoming.merge(inj_by_team.rename(columns={"team": "home_team", "count": "home_injured_count"}), on="home_team", how="left")
        upcoming = upcoming.merge(inj_by_team.rename(columns={"team": "away_team", "count": "away_injured_count"}), on="away_team", how="left")
        upcoming[["home_injured_count", "away_injured_count"]] = upcoming[["home_injured_count", "away_injured_count"]].fillna(0)
        upcoming["diff_injured_count"] = upcoming["home_injured_count"] - upcoming["away_injured_count"]

        inj_all = inj_df[(inj_df["report_status"].isin(["Out", "Doubtful"])) & (inj_df["week"] == target_week)].copy()
        inj_all["_status_wt"] = inj_all["report_status"].map(_STATUS_WEIGHT).fillna(0)
        inj_all["season"] = inj_all["season"].astype(int)
        allpro_hist = []
        for yrs_back, weight in zip([1, 2, 3], [4, 2, 1]):
            tmp = allpro_df.copy()
            tmp["season"] = tmp["Year"] + yrs_back
            tmp["weight"] = weight
            allpro_hist.append(tmp)
        allpro_wh = pd.concat(allpro_hist).drop_duplicates(["Player", "season"])
        allpro_wh = allpro_wh.copy()
        allpro_wh["_name_norm"] = allpro_wh["Player"].map(norm_name)
        inj_all["_name_norm"]   = inj_all["full_name"].map(norm_name)
        inj_all   = inj_all.merge(allpro_wh[["_name_norm", "season", "weight"]], on=["_name_norm", "season"], how="left")
        inj_all   = inj_all[inj_all["weight"].notnull()]
        inj_wt    = inj_all.groupby(["season", "week", "team"])["weight"].sum().reset_index().rename(columns={"weight": "inj_ap_wt"})
        upcoming  = upcoming.merge(inj_wt.rename(columns={"team": "home_team"}).drop(columns=["season", "week"]), on="home_team", how="left").rename(columns={"inj_ap_wt": "home_inj_ap_wt"})
        upcoming  = upcoming.merge(inj_wt.rename(columns={"team": "away_team"}).drop(columns=["season", "week"]), on="away_team", how="left").rename(columns={"inj_ap_wt": "away_inj_ap_wt"})
        upcoming[["home_inj_ap_wt", "away_inj_ap_wt"]] = upcoming[["home_inj_ap_wt", "away_inj_ap_wt"]].fillna(0)
        upcoming["home_active_allpro_weighted"] = (upcoming["home_allpro_last_3_years_weighted"] - upcoming["home_inj_ap_wt"]).clip(lower=0)
        upcoming["away_active_allpro_weighted"] = (upcoming["away_allpro_last_3_years_weighted"] - upcoming["away_inj_ap_wt"]).clip(lower=0)
        upcoming["diff_active_allpro_weighted"] = upcoming["home_active_allpro_weighted"] - upcoming["away_active_allpro_weighted"]
        _prev_yr_ap_norms   = set(allpro_df[allpro_df["Year"] == target_season - 1]["Player"].map(norm_name))
        inj_prev_yr         = inj_all[inj_all["_name_norm"].isin(_prev_yr_ap_norms)]
        inj_prev_yr_by_team = inj_prev_yr.groupby("team").size().reset_index(name="inj_ap_prev_yr")
        upcoming = upcoming.merge(inj_prev_yr_by_team.rename(columns={"team": "home_team", "inj_ap_prev_yr": "home_inj_ap_prev_yr"}), on="home_team", how="left")
        upcoming = upcoming.merge(inj_prev_yr_by_team.rename(columns={"team": "away_team", "inj_ap_prev_yr": "away_inj_ap_prev_yr"}), on="away_team", how="left")
        upcoming[["home_inj_ap_prev_yr", "away_inj_ap_prev_yr"]] = upcoming[["home_inj_ap_prev_yr", "away_inj_ap_prev_yr"]].fillna(0)
        _home_active_prev = (upcoming["home_allpro_prev_year"] - upcoming["home_inj_ap_prev_yr"]).clip(lower=0)
        _away_active_prev = (upcoming["away_allpro_prev_year"] - upcoming["away_inj_ap_prev_yr"]).clip(lower=0)
        upcoming["diff_active_allpro_prev_year"] = _home_active_prev - _away_active_prev

    except Exception as e:
        print(f"  ⚠️  Injury data unavailable: {e} — using zeros")
        for col in ["home_injured_count", "away_injured_count", "diff_injured_count", "diff_active_allpro_weighted", "diff_active_allpro_prev_year"]:
            upcoming[col] = 0
    return upcoming


In [ ]:
if RUN_TESTS:
    _fs = _synth_schedule()
    _allpro = _synth_allpro()
    _up = _fs[_fs["week"] == 5].copy()
    # Inject the Group 4 cols this helper depends on (we test Group 4 separately).
    _up = _build_allpro(_up, _allpro, target_season=2025)
    _out = _build_injuries(_up, _allpro, target_season=2025, target_week=5)
    for c in ["home_injured_count", "away_injured_count", "diff_injured_count",
              "diff_active_allpro_weighted", "diff_active_allpro_prev_year"]:
        assert c in _out.columns, f"Missing column {c}"
    print("✓ Group 9 — injuries: 5 cols present (offline fallback path).")


## Group 10 — `_build_coach_win_pct`

Career win% prior to this game + rolling 3-season win% for the home and away
head coach. Uses cumulative wins / games across the full coach history (1999+).

**Inputs:** `upcoming`, optional `coach_hist_df` (1999+ schedule with results),
`target_season`, `target_week`.
**Outputs:** 4 columns — `home/away_coach_win_pct_prior` + `..._roll3`.
**Test:** Andy Reid won every synth game → his roll3 should be 1.0.


In [ ]:
def _build_coach_win_pct(upcoming, coach_hist_df, target_season, target_week):
    """Group 10: career win% and rolling 3-season win% for home/away coach."""
    if coach_hist_df is None:
        _raw_coach = nfl.load_schedules(list(range(1999, target_season + 1)))
        _ch        = _raw_coach.to_pandas() if hasattr(_raw_coach, "to_pandas") else pd.DataFrame(_raw_coach)
        coach_hist = _ch[_ch["result"].notna()].copy()
    else:
        coach_hist = coach_hist_df
    home_c = coach_hist[["game_id", "season", "week", "home_team", "away_team", "home_score", "away_score", "home_coach"]].copy()
    home_c.rename(columns={"home_team": "team", "away_team": "opponent", "home_score": "team_score", "away_score": "opponent_score", "home_coach": "coach"}, inplace=True)
    away_c = coach_hist[["game_id", "season", "week", "away_team", "home_team", "away_score", "home_score", "away_coach"]].copy()
    away_c.rename(columns={"away_team": "team", "home_team": "opponent", "away_score": "team_score", "home_score": "opponent_score", "away_coach": "coach"}, inplace=True)
    games_df = pd.concat([home_c, away_c], ignore_index=True)
    games_df["win"] = (games_df["team_score"] > games_df["opponent_score"]).astype(int)

    def cumulative_coach(group):
        group = group.sort_values(["season", "week", "game_id"]).copy()
        group["cumulative_wins"]  = group["win"].cumsum().shift(fill_value=0)
        group["cumulative_games"] = group["win"].expanding().count().shift(fill_value=0)
        return group

    games_df = games_df.groupby("coach", group_keys=False).apply(cumulative_coach)
    games_df["coach_win_pct_prior"] = (
        games_df["cumulative_wins"] / games_df["cumulative_games"].replace(0, np.nan)
    ).fillna(0).round(3)
    latest_coach_wp = (
        games_df.sort_values(["season", "week", "game_id"])
        .groupby("coach").nth(-1).reset_index()[["coach", "coach_win_pct_prior"]]
    )
    upcoming = upcoming.merge(
        latest_coach_wp.rename(columns={"coach": "home_coach", "coach_win_pct_prior": "home_coach_win_pct_prior"}),
        on="home_coach", how="left"
    )
    upcoming = upcoming.merge(
        latest_coach_wp.rename(columns={"coach": "away_coach", "coach_win_pct_prior": "away_coach_win_pct_prior"}),
        on="away_coach", how="left"
    )
    _league_coach_avg = latest_coach_wp["coach_win_pct_prior"].mean() if len(latest_coach_wp) > 0 else 0.5
    upcoming[["home_coach_win_pct_prior", "away_coach_win_pct_prior"]] = (
        upcoming[["home_coach_win_pct_prior", "away_coach_win_pct_prior"]].fillna(_league_coach_avg)
    )

    # Rolling 3-season coach win% — more sensitive to recent performance than career win%
    _roll3_pool = games_df[
        ((games_df["season"] >= target_season - 3) & (games_df["season"] < target_season)) |
        ((games_df["season"] == target_season) & (games_df["week"] < target_week))
    ]
    _roll3_wp = (
        _roll3_pool.groupby("coach")
        .agg(wins=("win", "sum"), g=("win", "count"))
        .assign(coach_win_pct_roll3=lambda d: (d["wins"] / d["g"].replace(0, np.nan)).fillna(_league_coach_avg).round(3))
        .reset_index()[["coach", "coach_win_pct_roll3"]]
    )
    upcoming = upcoming.merge(
        _roll3_wp.rename(columns={"coach": "home_coach", "coach_win_pct_roll3": "home_coach_win_pct_roll3"}),
        on="home_coach", how="left"
    )
    upcoming = upcoming.merge(
        _roll3_wp.rename(columns={"coach": "away_coach", "coach_win_pct_roll3": "away_coach_win_pct_roll3"}),
        on="away_coach", how="left"
    )
    upcoming[["home_coach_win_pct_roll3", "away_coach_win_pct_roll3"]] = (
        upcoming[["home_coach_win_pct_roll3", "away_coach_win_pct_roll3"]].fillna(_league_coach_avg)
    )
    return upcoming


In [ ]:
if RUN_TESTS:
    _fs = _synth_schedule()
    _up = _fs[_fs["week"] == 5].copy()
    _coach_hist = _fs[_fs["result"].notna()].copy()
    _out = _build_coach_win_pct(_up, _coach_hist, target_season=2025, target_week=5)
    for c in ["home_coach_win_pct_prior", "away_coach_win_pct_prior",
              "home_coach_win_pct_roll3", "away_coach_win_pct_roll3"]:
        assert c in _out.columns, f"Missing column {c}"
    # Andy Reid (home_coach) won 4/4 synth games → 1.0
    assert _out["home_coach_win_pct_roll3"].iloc[0] == 1.0, \
        f"Andy Reid won 4/4 — expected 1.0, got {_out['home_coach_win_pct_roll3'].iloc[0]}"
    assert _out["away_coach_win_pct_roll3"].iloc[0] == 0.0, "Sean McDermott lost 4/4 → expected 0.0"
    print("✓ Group 10 — coach_win_pct: 4 cols present, win%s match synth ground truth.")


## Main pipeline — `build_features`

Top-level entry point that calls each per-group helper in order. Group 9
(injuries) depends on Group 4 (AllPro) columns, so they must run in that
sequence — this is enforced by the call order below, not by any assertion in
the helpers themselves.

**Inputs:**
- `target_week`, `target_season` — the game to predict.
- `full_schedule` — DataFrame of schedules ≥ `target_season - 1`.
- `pbp_rp` — DataFrame of run/pass PBP for the current + prior season.
- `allpro_df` — AllPro CSV DataFrame, team-mapped via `TEAM_MAP`.
- `week_margin_lkp` — optional pd.Series indexed by week; if None, derived
  from current-season history.
- `coach_hist_df` — optional 1999+ schedule with results; if None, Group 10
  fetches live via `nflreadpy`.
- `required_features` — iterable of features that must be present at the end.
  Missing features are zero-filled with a warning. Defaults to `FEATURE_COLS_85`.

**Output:** the `upcoming` DataFrame with all 85 feature columns appended,
NaNs imputed with column-median. Returns `None` if no games for that week.

**Test:** synthetic-data integration test — asserts the result has all 85
expected features and none of the headline columns are NaN.


In [ ]:
def build_features(
    target_week,
    target_season,
    full_schedule,
    pbp_rp,
    allpro_df,
    week_margin_lkp=None,
    coach_hist_df=None,
    required_features: Optional[Iterable[str]] = None,
):
    """Build all 85 features for ``target_week`` using only data available
    before that week. Returns the ``upcoming`` DataFrame with feature columns
    appended, or ``None`` if no games for that week."""
    history = full_schedule[
        (full_schedule["season"] == target_season) &
        (full_schedule["week"]   <  target_week) &
        (full_schedule["result"].notna())
    ].copy()

    upcoming = full_schedule[
        (full_schedule["season"] == target_season) &
        (full_schedule["week"]   == target_week)
    ].copy()

    if upcoming.empty:
        return None

    if history.empty:
        print(f"  ℹ️  Week 1: no season history yet — SOS/scoring/cover features will be zero-filled")

    # ── Shared intermediates used by multiple groups ───────────────────────────
    pbp_s = pbp_rp[
        ((pbp_rp["season"] == target_season) & (pbp_rp["week"] < target_week)) |
        (pbp_rp["season"] == target_season - 1)
    ].copy()
    wk_lookup = pbp_rp[["game_id", "week", "season"]].drop_duplicates()

    _req_cols = ["season", "week", "home_team", "away_team", "home_score", "away_score", "result", "spread_line"]
    if coach_hist_df is not None and all(c in coach_hist_df.columns for c in _req_cols):
        _hist_rolling = coach_hist_df[
            ((coach_hist_df["season"] == target_season) & (coach_hist_df["week"] < target_week)) |
            (coach_hist_df["season"] == target_season - 1)
        ][_req_cols].copy()
    else:
        _hist_rolling = history[[c for c in _req_cols if c in history.columns]].copy()

    # ── Apply feature groups in order ─────────────────────────────────────────
    upcoming = _build_schedule_context(upcoming, full_schedule, target_week)
    upcoming = _build_rolling_pbp(upcoming, pbp_s, wk_lookup)
    upcoming = _build_sos_and_performance(upcoming, _hist_rolling, history, week_margin_lkp)
    upcoming = _build_allpro(upcoming, allpro_df, target_season)               # Group 4 must run before Group 9
    upcoming = _build_situational_pbp(upcoming, pbp_s, wk_lookup)
    upcoming = _build_qb_switch(upcoming, history, coach_hist_df, target_season)
    upcoming = _build_passer_rating(upcoming, pbp_rp, target_season)
    upcoming = _build_injuries(upcoming, allpro_df, target_season, target_week)  # depends on Group 4 columns
    upcoming = _build_coach_win_pct(upcoming, coach_hist_df, target_season, target_week)

    # ── Final feature check ───────────────────────────────────────────────────
    _all_required = list(dict.fromkeys(required_features)) if required_features is not None else list(FEATURE_COLS_85)
    missing = [f for f in _all_required if f not in upcoming.columns]
    if missing:
        print(f"  ⚠️  {len(missing)} features missing for week {target_week}: {missing}")
        for m in missing:
            upcoming[m] = 0

    _nan_cols = [c for c in upcoming.select_dtypes(include="number").columns
                 if upcoming[c].isna().any()]
    if _nan_cols:
        print(f"  ⚠️  {len(_nan_cols)} NaN column(s) imputed with median: {_nan_cols}")
    upcoming = upcoming.fillna(upcoming.median(numeric_only=True).fillna(0))
    return upcoming


In [ ]:
if RUN_TESTS:
    _res = build_features(
        target_week=5,
        target_season=2025,
        full_schedule=_synth_schedule(),
        pbp_rp=_synth_pbp(),
        allpro_df=_synth_allpro(),
        week_margin_lkp=None,
        coach_hist_df=_synth_schedule()[_synth_schedule()["result"].notna()].copy(),
    )
    assert _res is not None, "build_features returned None on non-empty synth input"
    assert isinstance(_res, pd.DataFrame), "build_features did not return a DataFrame"
    assert len(_res) == 1, f"Expected 1 upcoming row, got {len(_res)}"
    _missing = [c for c in FEATURE_COLS_85 if c not in _res.columns]
    assert not _missing, f"FEATURE_COLS_85 not all present after build_features: {_missing}"
    for _c in ["home_rolling_win_pct", "home_coach_win_pct_prior", "home_coach_win_pct_roll3",
               "sos_diff", "cover_rate_diff",
               "home_pr_prev_year", "home_cpae_prev_year", "home_time_to_throw_prev_year",
               "diff_pr_prev_year", "diff_cpae_prev_year", "diff_time_to_throw_prev_year"]:
        assert not pd.isna(_res[_c].iloc[0]), f"Column {_c} is NaN"
    print(f"✓ build_features integration: 1 row, {len(_res.columns)} cols, all 85 features present.")


## `build_numeric_features` — categorical encode + numeric matrix

Used by the Ensemble and LightGBM voters. Takes the `upcoming` DataFrame,
ordinally encodes `roof` and `surface` against an already-fit `OrdinalEncoder`,
then assembles a float32 numeric matrix in the order specified by
`feature_cols`. Unknown / NaN categories fall back to the encoder's first known
category so the matrix stays dense.

**Inputs:** `upcoming_df`, `feature_cols` (ordered list), `enc` (fit OrdinalEncoder).
**Output:** float32 numpy array of shape `(len(upcoming_df), len(feature_cols))`.
**Test:** known-categories produces correct shape/dtype; unknown and NaN
inputs don't raise.


In [ ]:
def build_numeric_features(upcoming_df, feature_cols, enc):
    """Build an ordinal-encoded numeric feature matrix for the Ensemble and
    LightGBM voters. ``enc`` is a fit OrdinalEncoder over (roof, surface).
    Unknown / NaN categories fall back to the first known category to keep the
    matrix dense.
    """
    df = upcoming_df.copy()
    if hasattr(enc, "categories_"):
        for i, col in enumerate(["roof", "surface"]):
            known    = set(enc.categories_[i])
            fallback = enc.categories_[i][0]
            df[col]  = df[col].fillna(fallback).apply(lambda v: v if v in known else fallback)
    df[["roof", "surface"]] = enc.transform(df[["roof", "surface"]])
    X = np.zeros((len(df), len(feature_cols)), dtype="float32")
    for i, col in enumerate(feature_cols):
        if col in df.columns:
            X[:, i] = pd.to_numeric(df[col], errors="coerce").fillna(0).values
    return X


In [ ]:
if RUN_TESTS:
    from sklearn.preprocessing import OrdinalEncoder
    _enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    _enc.fit([["dome", "turf"], ["outdoors", "grass"], ["retractable", "turf"]])
    _feats = ["roof", "surface", "is_playoff", "spread_line"]

    # Known categories → correct shape + dtype
    _df_ok = pd.DataFrame({"roof": ["dome"], "surface": ["turf"], "is_playoff": [0], "spread_line": [-3.0]})
    _X = build_numeric_features(_df_ok, _feats, _enc)
    assert isinstance(_X, np.ndarray), "Expected numpy array"
    assert _X.shape == (1, 4),          f"Expected shape (1,4), got {_X.shape}"
    assert np.issubdtype(_X.dtype, np.floating), "Expected float dtype"

    # Unknown category → must not raise
    _df_unk = pd.DataFrame({"roof": ["open_air_xyz"], "surface": ["sod"], "is_playoff": [0], "spread_line": [2.5]})
    build_numeric_features(_df_unk, _feats, _enc)

    # NaN input → must not raise
    _df_nan = pd.DataFrame({"roof": [None], "surface": [None], "is_playoff": [0], "spread_line": [-1.0]})
    build_numeric_features(_df_nan, _feats, _enc)
    print("✓ build_numeric_features: known + unknown + NaN inputs all pass.")


## Namespace cleanup

When this notebook is loaded by a consumer (e.g. ``predict_betting.ipynb`` or
``model_comparison.ipynb``) with ``RUN_TESTS = False``, the synthetic-fixture
helpers have no business sitting in the consumer’s global namespace. The cell
below removes them. When run standalone for development, the fixtures stay
accessible for ad-hoc debugging.


In [ ]:
# Remove test scaffolding from consumer namespaces (when imported with RUN_TESTS=False).
# Run standalone (RUN_TESTS=True), the synth helpers stay around for ad-hoc debugging.
if not RUN_TESTS:
    for _name in ("_synth_schedule", "_synth_pbp", "_synth_allpro"):
        globals().pop(_name, None)


## Summary

All public names are now in the kernel namespace and ready for use by consumer
notebooks (which load this one by json+exec'ing each code cell into their own globals). If you reached this cell with no failed
assertions, the feature engineering is sound on synthetic data — production
correctness is then validated against live nflreadpy data in the consumer
notebook's own test cells.

**Don't change a feature name without retraining the production pkls.** The
trailing space in `allpro_diff_home_def_away_off_3_years ` is intentional and
matches the column the model was trained on.
